In [4]:
import pandas as pd

df_prep = pd.read_csv('/content/hasil_praproses_korpus_ire.csv')

print("Jumlah dokumen:", len(df_prep))
print("Kolom:", df_prep.columns.tolist())

df_prep.head()

Jumlah dokumen: 50
Kolom: ['No', 'Judul', 'Isi', 'Sumber', 'Tahun', 'URL', 'hasil_preprocessing']


,No,Judul,Isi,Sumber,Tahun,URL,hasil_preprocessing
0,1,PT Timah Masih Lebur Lagi Logam dari Smelter S...,"JAKARTA, KOMPAS.com - PT Timah Tbk disebut mel...",Kompas,2024,https://nasional.kompas.com/read/2024/09/18/19...,jakarta kompas com pt timah tbk sebut lebur pr...
1,2,Paloh Doakan Prabowo: Semoga Pemerintah di Bag...,Ketum Partai NasDem Surya Paloh mendoakan Pres...,Detik,2024,https://news.detik.com/berita/d-7512004/paloh-...,tum partai nasdem surya paloh doa presiden pil...
2,3,"Tarif Impor AS Diberlakukan atas Cina, Meksiko...","Baca berita dengan sedikit iklan, klik di sin...",Tempo,2025,https://www.tempo.co/internasional/tarif-impor...,baca berita sedikit iklan klik sini tarif impo...
3,4,Pramono soal Bestari Barus NasDem Muncul di Ag...,"Cagub Jakarta nomor urut 3, Pramono Anung, men...",Detik,2024,https://news.detik.com/pilkada/d-7585172/pramo...,cagub jakarta nomor urut pramono anung tanggap...
4,5,Ketua GRIB Jaya Tangsel yang Duduki Lahan BMKG...,"JAKARTA, KOMPAS.com - Ketua Dewan Pimpinan Cab...",Kompas,2025,https://megapolitan.kompas.com/read/2025/05/26...,jakarta kompas com ketua dewan pimpin cabang d...


In [6]:
queries = [
    "timah logam",
    "pemerintah presiden",
    "tarif impor",
    "jakarta pilkada",
    "nasdem partai",
    "ekonomi indonesia",
    "perdagangan internasional",
    "kebijakan pemerintah",
    "masyarakat hukum",
    "politik nasional"
]

for i, q in enumerate(queries, 1):
    print(f"{i}. {q}")

1. timah logam
2. pemerintah presiden
3. tarif impor
4. jakarta pilkada
5. nasdem partai
6. ekonomi indonesia
7. perdagangan internasional
8. kebijakan pemerintah
9. masyarakat hukum
10. politik nasional


In [7]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df_prep["hasil_preprocessing"] = (
    df_prep["hasil_preprocessing"]
    .fillna("")
    .astype(str)
)

documents = df_prep["hasil_preprocessing"].tolist()

print("Jumlah dokumen:", len(documents))
print("Contoh teks:", documents[0][:200])

Jumlah dokumen: 50
Contoh teks: jakarta kompas com pt timah tbk sebut lebur produk logam timah belum olah smelter milik lima usaha swasta ungkap jaksa tuntut umum jpu gali terang mantan evaluator devisa pt timah apit rinaldi susanto


In [8]:
vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False
)

tfidf_matrix = vectorizer.fit_transform(documents)

def cari_tfidf(query, top_n=10):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

    urutan = np.argsort(scores)[::-1][:top_n]

    hasil = df_prep.iloc[urutan][["No", "Judul"]].copy()
    hasil["Skor_TFIDF"] = scores[urutan]
    hasil["Peringkat"] = range(1, len(hasil) + 1)

    return hasil[["Peringkat", "No", "Judul", "Skor_TFIDF"]]

cari_tfidf("timah logam")

,Peringkat,No,Judul,Skor_TFIDF
0,1,1,PT Timah Masih Lebur Lagi Logam dari Smelter S...,0.618660
44,2,45,Budi Said akan Divonis 27 Desember di Kasus Ma...,0.062146
7,3,8,"MA Kabulkan PK Antam Lawan ""Crazy Rich"" Budi Said",0.027184
40,4,41,Indonesia Tegaskan Posisi di Kancah Global Lew...,0.019617
48,5,49,PSN Merusak Lingkungan. Bisakah Disebut Melang...,0.000000
49,6,50,"Dipimpin Irwasum, Polri Evaluasi Penggunaan Se...",0.000000
45,7,46,Disdik Depok Minta Ortu Siswa Daftar Lebih Awa...,0.000000
43,8,44,Menteri Imipas Beri Ultimatum Saat Lantik Seju...,0.000000
46,9,47,"Pesawat Kecil Tabrak Bus di Sao Paulo, 2 Orang...",0.000000
47,10,48,"Pasar Taman Puring Kebakaran Hari Ini, Pedagan...",0.000000


In [10]:
import math
from collections import Counter

token_docs = [doc.split() for doc in documents]

N = len(token_docs)
avgdl = sum(len(doc) for doc in token_docs) / N

doc_freqs = [Counter(doc) for doc in token_docs]

df_terms = Counter()
for freqs in doc_freqs:
    for term in freqs:
        df_terms[term] += 1

def cari_bm25(query, k1=1.2, b=0.75, top_n=10):
    query_terms = query.split()
    scores = []

    for i, freqs in enumerate(doc_freqs):
        dl = len(token_docs[i])
        score = 0.0

        for term in query_terms:
            tf = freqs.get(term, 0)

            if tf == 0:
                continue

            df_t = df_terms[term]
            idf = math.log(
                1 + (N - df_t + 0.5) / (df_t + 0.5)
            )

            pembagi = tf + k1 * (
                1 - b + b * dl / avgdl
            )

            score += idf * (tf * (k1 + 1)) / pembagi

        scores.append(score)

    urutan = sorted(
        range(N), key=lambda i: scores[i], reverse=True
    )[:top_n]

    hasil = df_prep.iloc[urutan][["No", "Judul"]].copy()
    hasil["Skor_BM25"] = [scores[i] for i in urutan]
    hasil["Peringkat"] = range(1, len(hasil) + 1)

    return hasil[["Peringkat", "No", "Judul", "Skor_BM25"]]

In [11]:
cari_bm25("timah logam")

,Peringkat,No,Judul,Skor_BM25
0,1,1,PT Timah Masih Lebur Lagi Logam dari Smelter S...,11.203576
44,2,45,Budi Said akan Divonis 27 Desember di Kasus Ma...,3.552496
7,3,8,"MA Kabulkan PK Antam Lawan ""Crazy Rich"" Budi Said",2.777006
40,4,41,Indonesia Tegaskan Posisi di Kancah Global Lew...,1.917058
1,5,2,Paloh Doakan Prabowo: Semoga Pemerintah di Bag...,0.000000
2,6,3,"Tarif Impor AS Diberlakukan atas Cina, Meksiko...",0.000000
3,7,4,Pramono soal Bestari Barus NasDem Muncul di Ag...,0.000000
4,8,5,Ketua GRIB Jaya Tangsel yang Duduki Lahan BMKG...,0.000000
5,9,6,KPU DKI Masih Buka Layanan Pindah Memilih bagi...,0.000000
6,10,7,Pramono Anung Siapkan 2 Saksi per TPS di Pilka...,0.000000


In [12]:
query_uji = "timah logam"

hasil_tfidf = cari_tfidf(query_uji, top_n=10)
hasil_bm25 = cari_bm25(query_uji, top_n=10)

print("HASIL TF-IDF")
display(hasil_tfidf)

print("HASIL BM25")
display(hasil_bm25)

HASIL TF-IDF


,Peringkat,No,Judul,Skor_TFIDF
0,1,1,PT Timah Masih Lebur Lagi Logam dari Smelter S...,0.618660
44,2,45,Budi Said akan Divonis 27 Desember di Kasus Ma...,0.062146
7,3,8,"MA Kabulkan PK Antam Lawan ""Crazy Rich"" Budi Said",0.027184
40,4,41,Indonesia Tegaskan Posisi di Kancah Global Lew...,0.019617
48,5,49,PSN Merusak Lingkungan. Bisakah Disebut Melang...,0.000000
49,6,50,"Dipimpin Irwasum, Polri Evaluasi Penggunaan Se...",0.000000
45,7,46,Disdik Depok Minta Ortu Siswa Daftar Lebih Awa...,0.000000
43,8,44,Menteri Imipas Beri Ultimatum Saat Lantik Seju...,0.000000
46,9,47,"Pesawat Kecil Tabrak Bus di Sao Paulo, 2 Orang...",0.000000
47,10,48,"Pasar Taman Puring Kebakaran Hari Ini, Pedagan...",0.000000


HASIL BM25


,Peringkat,No,Judul,Skor_BM25
0,1,1,PT Timah Masih Lebur Lagi Logam dari Smelter S...,11.203576
44,2,45,Budi Said akan Divonis 27 Desember di Kasus Ma...,3.552496
7,3,8,"MA Kabulkan PK Antam Lawan ""Crazy Rich"" Budi Said",2.777006
40,4,41,Indonesia Tegaskan Posisi di Kancah Global Lew...,1.917058
1,5,2,Paloh Doakan Prabowo: Semoga Pemerintah di Bag...,0.000000
2,6,3,"Tarif Impor AS Diberlakukan atas Cina, Meksiko...",0.000000
3,7,4,Pramono soal Bestari Barus NasDem Muncul di Ag...,0.000000
4,8,5,Ketua GRIB Jaya Tangsel yang Duduki Lahan BMKG...,0.000000
5,9,6,KPU DKI Masih Buka Layanan Pindah Memilih bagi...,0.000000
6,10,7,Pramono Anung Siapkan 2 Saksi per TPS di Pilka...,0.000000


In [13]:
for k1, b in [
    (0.5, 0.25),
    (1.2, 0.75),
    (2.0, 0.75),
    (1.2, 1.0)
]:
    print(f"\nk1={k1}, b={b}")
    display(cari_bm25("timah logam", k1=k1, b=b, top_n=5))


k1=0.5, b=0.25


,Peringkat,No,Judul,Skor_BM25
0,1,1,PT Timah Masih Lebur Lagi Logam dari Smelter S...,8.297416
44,2,45,Budi Said akan Divonis 27 Desember di Kasus Ma...,2.944886
7,3,8,"MA Kabulkan PK Antam Lawan ""Crazy Rich"" Budi Said",2.491581
40,4,41,Indonesia Tegaskan Posisi di Kancah Global Lew...,2.302787
1,5,2,Paloh Doakan Prabowo: Semoga Pemerintah di Bag...,0.000000



k1=1.2, b=0.75


,Peringkat,No,Judul,Skor_BM25
0,1,1,PT Timah Masih Lebur Lagi Logam dari Smelter S...,11.203576
44,2,45,Budi Said akan Divonis 27 Desember di Kasus Ma...,3.552496
7,3,8,"MA Kabulkan PK Antam Lawan ""Crazy Rich"" Budi Said",2.777006
40,4,41,Indonesia Tegaskan Posisi di Kancah Global Lew...,1.917058
1,5,2,Paloh Doakan Prabowo: Semoga Pemerintah di Bag...,0.000000



k1=2.0, b=0.75


,Peringkat,No,Judul,Skor_BM25
0,1,1,PT Timah Masih Lebur Lagi Logam dari Smelter S...,14.122848
44,2,45,Budi Said akan Divonis 27 Desember di Kasus Ma...,3.960212
7,3,8,"MA Kabulkan PK Antam Lawan ""Crazy Rich"" Budi Said",2.868716
40,4,41,Indonesia Tegaskan Posisi di Kancah Global Lew...,1.831446
1,5,2,Paloh Doakan Prabowo: Semoga Pemerintah di Bag...,0.000000



k1=1.2, b=1.0


,Peringkat,No,Judul,Skor_BM25
0,1,1,PT Timah Masih Lebur Lagi Logam dari Smelter S...,11.214254
44,2,45,Budi Said akan Divonis 27 Desember di Kasus Ma...,3.630194
7,3,8,"MA Kabulkan PK Antam Lawan ""Crazy Rich"" Budi Said",2.916881
40,4,41,Indonesia Tegaskan Posisi di Kancah Global Lew...,1.791444
1,5,2,Paloh Doakan Prabowo: Semoga Pemerintah di Bag...,0.000000
